# Implement Minibatch KMeans with Atlas Streams

This tutorial implements a small minibatch KMeans-style algorithm using
scAtlasPy expression streams. It demonstrates how an iterative method can make
repeated passes over atlas-scale data without materializing the complete
cell-by-gene matrix in memory.

The implementation is intentionally simple and is intended for learning how to
integrate custom methods with scAtlasPy. For routine clustering, use the
built-in `sap.tl.kmeans()` function.

By the end of this tutorial, you will be able to:

- obtain randomized, multi-pass expression minibatches from an Atlas;
- initialize and update model parameters incrementally;
- calculate distances without creating large three-dimensional arrays;
- monitor a custom training loop;
- save the fitted parameters for later full-Atlas assignment.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- the expression representation intended for clustering is available;
- the cells and genes used for training have been selected;
- the resulting dense minibatches fit comfortably in memory.

Import the required packages:



In [ ]:
import os
from pathlib import Path

import numpy as np
import scatlaspy as sap



The examples below assume that an existing Atlas is already open:



In [ ]:
os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



## 1. Define the Training View

Build a read index that defines the cells, genes, and expression field supplied
to the algorithm:



In [ ]:
atlas.build_read_index(
    cell_condition="filter_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)



In this example, the algorithm uses:

- cells selected by `filter_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The fitted centroids are meaningful only for this exact feature set, feature
order, and expression representation. Save this information alongside the
model parameters if the centroids will be reused in another session.
```

## 2. Configure the Training Stream

KMeans is an iterative algorithm. Rather than traversing the selected cells
only once in a fixed order, training should draw randomized minibatches over
multiple passes.

Create a multi-pass expression stream:



In [ ]:
batch_size = 2048
max_batches = 1000

batch_iter = atlas.get_minibatch_dense(
    pass_mode="multi-pass",
    batch_size=batch_size,
    buffer_batch_num=5,
    max_batches=max_batches,
)



The main parameters are:

| Parameter | Purpose |
|---|---|
| `pass_mode="multi-pass"` | Allows repeated access to the selected expression data during training. |
| `batch_size` | Controls the number of cells in each dense minibatch. |
| `buffer_batch_num` | Controls the size of the buffer used to mix batches before they are yielded. |
| `max_batches` | Sets the maximum number of minibatches used for fitting. |

A larger `batch_size` may improve computational throughput but increases the
memory required for each dense expression matrix. A larger shuffle buffer may
improve mixing but also requires additional memory.

```{note}
A multi-pass stream is intended for model fitting. Because cells may be
randomized and revisited, it should not be used when prediction outputs must be
aligned directly with cell order. Use a deterministic single-pass stream for
full-Atlas assignment.
```

## 3. Initialize the Centroids

Set the number of clusters and random-number generator:



In [ ]:
n_clusters = 10
rng = np.random.default_rng(42)



Retrieve the first minibatch:



In [ ]:
try:
    first_batch = next(batch_iter)
except StopIteration as exc:
    raise ValueError("The current read index contains no expression data.") from exc

first_batch = np.asarray(first_batch, dtype=np.float32)



Confirm that the first minibatch contains enough cells:



In [ ]:
if first_batch.shape[0] < n_clusters:
    raise ValueError(
        "The first minibatch contains fewer cells than n_clusters."
    )



Initialize the centroids from randomly selected cells:



In [ ]:
init_idx = rng.choice(
    first_batch.shape[0],
    size=n_clusters,
    replace=False,
)

centroids = first_batch[init_idx].copy()
counts = np.zeros(n_clusters, dtype=np.int64)



`counts[k]` records how many cells have contributed to centroid `k`. It is used
to calculate the incremental update weight.

This random initialization is sufficient for demonstrating the streaming
workflow. Production KMeans implementations commonly use more robust
initialization strategies such as KMeans++.

## 4. Calculate Assignments Efficiently

A direct broadcasting implementation such as:



In [ ]:
X_batch[:, None, :] - centroids[None, :, :]



creates an array with shape:

```text
n_cells_in_batch × n_clusters × n_features
```

This temporary array can be much larger than the input minibatch.

Instead, calculate squared Euclidean distances using:

\[
\lVert x-c\rVert^2
=
\lVert x\rVert^2
-2x^\mathsf{T}c
+\lVert c\rVert^2.
\]

Define a helper function:



In [ ]:
def assign_clusters(
    X_batch: np.ndarray,
    centroids: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Assign each row of X_batch to its nearest centroid."""

    sample_norm = np.sum(X_batch * X_batch, axis=1, keepdims=True)
    centroid_norm = np.sum(
        centroids * centroids,
        axis=1,
        keepdims=True,
    ).T

    squared_distances = (
        sample_norm
        - 2.0 * X_batch @ centroids.T
        + centroid_norm
    )

    # Floating-point roundoff can produce very small negative values.
    np.maximum(squared_distances, 0.0, out=squared_distances)

    assignments = np.argmin(squared_distances, axis=1)
    minimum_distances = squared_distances[
        np.arange(X_batch.shape[0]),
        assignments,
    ]

    return assignments, minimum_distances



This implementation creates a distance matrix with shape:

```text
n_cells_in_batch × n_clusters
```

rather than a three-dimensional cell-by-cluster-by-feature array.

## 5. Define the Incremental Update

For each cluster, combine the previous centroid with the mean of the cells
assigned to that cluster in the current minibatch:



In [ ]:
def update_centroids(
    X_batch: np.ndarray,
    assignments: np.ndarray,
    centroids: np.ndarray,
    counts: np.ndarray,
) -> None:
    """Update centroids in place using one minibatch."""

    for cluster_id in range(centroids.shape[0]):
        member_mask = assignments == cluster_id
        n_members = int(member_mask.sum())

        if n_members == 0:
            continue

        batch_center = X_batch[member_mask].mean(axis=0)

        old_count = counts[cluster_id]
        new_count = old_count + n_members
        weight = n_members / new_count

        centroids[cluster_id] = (
            (1.0 - weight) * centroids[cluster_id]
            + weight * batch_center
        )

        counts[cluster_id] = new_count



This update treats the new centroid as the cumulative mean of all cells assigned
to the cluster during training.

Clusters that receive no cells in a minibatch are left unchanged.

```{note}
This educational implementation does not reinitialize persistently empty
clusters. A production implementation should detect clusters that receive few
or no assignments and apply an appropriate reinitialization strategy.
```

## 6. Run the Training Loop

Use the first batch for both initialization and the first model update, then
continue consuming the remaining minibatches.

Define a small helper that yields the first batch followed by the rest of the
stream:



In [ ]:
from itertools import chain

training_batches = chain([first_batch], batch_iter)



Run the minibatch updates:



In [ ]:
inertia_history = []

for batch_id, X_batch in enumerate(training_batches, start=1):
    X_batch = np.asarray(X_batch, dtype=np.float32)

    if X_batch.ndim != 2:
        raise ValueError("Each minibatch must be a two-dimensional matrix.")

    if X_batch.shape[1] != centroids.shape[1]:
        raise ValueError(
            "The minibatch feature dimension does not match the centroids."
        )

    assignments, minimum_distances = assign_clusters(
        X_batch,
        centroids,
    )

    batch_inertia = float(minimum_distances.mean())
    inertia_history.append(batch_inertia)

    update_centroids(
        X_batch,
        assignments,
        centroids,
        counts,
    )

    if batch_id == 1 or batch_id % 100 == 0:
        print(
            f"Batch {batch_id:>4}: "
            f"mean squared distance = {batch_inertia:.4f}"
        )



Inspect the fitted parameters:



In [ ]:
print("Centroid shape:", centroids.shape)
print("Assigned cells per cluster:", counts)



The batch inertia is calculated before each update. Because different randomized
minibatches are used, it will generally fluctuate rather than decrease
monotonically.

A decreasing or stabilizing trend can indicate that the model is learning, but
a fixed `max_batches` value does not guarantee convergence.

## 7. Inspect the Training Diagnostics

Summarize the initial and final parts of the training history:



In [ ]:
window = min(50, len(inertia_history))

initial_inertia = np.mean(inertia_history[:window])
final_inertia = np.mean(inertia_history[-window:])

print(f"Initial mean batch inertia: {initial_inertia:.4f}")
print(f"Final mean batch inertia:   {final_inertia:.4f}")



Also check whether all clusters received assignments:



In [ ]:
empty_clusters = np.flatnonzero(counts == 0)

if len(empty_clusters) > 0:
    print("Clusters without assignments:", empty_clusters.tolist())
else:
    print("All clusters received assignments.")



These checks are diagnostic rather than formal convergence tests. A more
complete implementation could include:

- a validation stream;
- centroid-displacement monitoring;
- early stopping;
- KMeans++ initialization;
- empty-cluster reinitialization;
- multiple independent initializations.

## 8. Save the Fitted Parameters

Save the centroids and assignment counts:



In [ ]:
output_path = Path("./results/minibatch_kmeans_model.npz")
output_path.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    output_path,
    centroids=centroids,
    counts=counts,
    n_clusters=n_clusters,
)

print(f"Saved model parameters to {output_path}")



The saved arrays are not sufficient by themselves to reproduce inference.
Record the associated:

- Atlas or dataset version;
- selected cells and genes;
- exact gene order;
- expression field;
- normalization and scaling configuration;
- random seed;
- training-stream parameters.

A model must receive features in the same order and representation used during
training.

## 9. Compare with the Built-in Tool

For routine clustering, use the built-in implementation:



In [ ]:
sap.tl.kmeans(
    atlas,
    n_clusters=10,
    fit_batches=1000,
    batch_size=2048,
    buffer_batch_num=5,
)



The built-in function manages model fitting, full-dataset assignment, and result
storage through the scAtlasPy workflow.

Use the custom pattern when you want to:

- prototype a new iterative algorithm;
- modify the parameter-update rule;
- integrate another machine-learning library;
- control the training loop directly;
- investigate alternative clustering objectives.

## Limitations of This Example

The implementation in this tutorial is designed to demonstrate data access and
incremental parameter updates. It is not a complete replacement for a
production-quality KMeans implementation.

In particular, it does not provide:

- KMeans++ initialization;
- multiple random restarts;
- robust empty-cluster handling;
- formal convergence criteria;
- automatic parameter tuning;
- direct writing of final cluster labels to `obs`.



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [ ]:
atlas.close()


## Next Steps

Continue with {doc}`apply-model-to-full-atlas` to assign every cell selected by
the read index to its nearest fitted centroid using a deterministic
single-pass stream.

For production clustering with automatic result storage, use
`sap.tl.kmeans()`.